# Predicting irrigation need - preprocessing

Feature engineering for the *Predicting Irrigation Need* competition. The competition data is **synthetically generated** from a smaller labeled `original_dataset`, so most features here try to recover artifacts of that generation process — digit/rounding patterns, value snapping, resampling ratios — on top of standard categorical interactions.

**Inputs** (in `./data/`):
- `train.csv`, `test.csv` — competition data, indexed by `id`.
- `original_dataset.csv` — the labeled source data the competition set was generated from. Optional: set `original_data = None` to skip the features that depend on it.

**Output:** `train_x_preprocessed.csv` / `test_x_preprocessed.csv` — the same rows with the engineered feature columns added, ready for the modeling notebooks.

The pipeline is configured in one cell (below) and applied in two groups: features that need only `train`/`test`, then features that additionally use the original dataset.

In [21]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Configuration

All dataset-specific settings live in the next cell: data location, target column, and the radix interaction specs. To run this notebook on a different dataset, edit only that cell.

In [22]:
# Everything dataset-specific lives here, so the feature functions below stay
# generic. To reuse this pipeline on another dataset, edit only this cell.
DATA_DIR = './data'
TRAIN_DATA = 'train_example.csv'
TEST_DATA = 'test_example.csv'
ORIG_DATA = 'orig_example.csv'  # optional, for distance features only

# Here is an example of a target column name; change it to your dataset's target column.
TARGET = 'Irrigation_Need'

# Radix interaction features. Each entry produces a column `radix_<name>` equal
# to the (optionally rounded) sum of its terms. A term contributes either a raw
# numeric value (kind='num') or a deterministic integer category code
# (kind='cat'), scaled by `mult` and optionally cast to int. Retarget radix
# encoding to new data by editing this dict only — no code changes needed.
# Here is an example of a radix feature that combines two categorical columns into a single integer code:
RADIX_FEATURES = {
    'Soil_pH_Soil_Type': {
        'round': None,
        'terms': [
            {'col': 'Soil_pH',   'kind': 'num', 'mult': 100, 'cast_int': True},
            {'col': 'Soil_Type', 'kind': 'cat', 'mult': 1000},
        ],
    },
    'Crop_Type_Crop_Growth_Stage': {
        'round': None,
        'terms': [
            {'col': 'Crop_Type',         'kind': 'cat', 'mult': 100},
            {'col': 'Crop_Growth_Stage', 'kind': 'cat', 'mult': 1},
        ],
    },
    'Temperature_C_Humidity': {
        'round': 3,
        'terms': [
            {'col': 'Temperature_C', 'kind': 'num', 'mult': 100},
            {'col': 'Humidity',      'kind': 'num', 'mult': 1},
        ],
    },
}

# Target-aware features (Group 3) — Ordered Target Encoding.
# TE_COLS=None falls back to the detected categorical_cols at pipeline time;
# set an explicit list to also target-encode e.g. the bigram columns.
TE_COLS = None
TE_N_SHUFFLES = 4   # ordered-TE passes averaged per row (variance reduction)
TE_SMOOTHING = 1    # additive smoothing toward the global class prior
TE_SEED = 2026      # shuffle seed for the ordered passes

## Load & prepare data

Read the competition files (and the optional original dataset), split off the target, and detect numeric vs. categorical columns. These raw column lists drive every feature function below.

In [23]:
train_data = pd.read_csv(f'{DATA_DIR}/{TRAIN_DATA}', index_col='id')
test_data = pd.read_csv(f'{DATA_DIR}/{TEST_DATA}', index_col='id')

# Optional: the labeled source dataset the competition data was generated from.
# Set to None when reusing this pipeline on data without such a reference — the
# Group 2 features then skip automatically.
original_data = pd.read_csv(f'{DATA_DIR}/{ORIG_DATA}', index_col=0)

In [24]:
train_x = train_data.drop(columns=[TARGET])
train_y = train_data[TARGET]
test_x = test_data.copy()

In [25]:
numeric_cols = test_data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = test_data.select_dtypes(include='object').columns.tolist()

C:\Users\Evgenii\AppData\Local\Temp\ipykernel_16708\4257074289.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = test_data.select_dtypes(include='object').columns.tolist()


## Feature engineering

Every function takes `train_x`/`test_x`, derives whatever maps/stats it needs, and returns copies with new columns; the shared `apply_to_both` helper handles the copy-and-apply boilerplate so each function only defines a single `mutate(df)` closure.

Nothing here is hardcoded to this dataset: the numeric/categorical column lists are detected with `select_dtypes`, the target name comes from `TARGET`, and the radix interactions are described by the editable `RADIX_FEATURES` config.

The functions are split into two groups by whether they need the original dataset:

- **Group 1** decimal/digit features, categorical bigrams, frequency encoding, radix interactions. Need only `train`/`test`, so they always run.
- **Group 2** KNN lookup, snap features, count-ratio encoding. Compare against `original_data` and are skipped when it is `None`.

In [26]:
def apply_to_both(train_x, test_x, mutate):
    train_out, test_out = train_x.copy(), test_x.copy()
    mutate(train_out)
    mutate(test_out)
    return train_out, test_out

## Group 1: features that need only train/test

These run on any dataset; they never touch `original_data`.

### Decimal / digit features — `add_decimal_features`

Decomposes each numeric value into **digit-level components**, which can expose patterns left by the synthetic data generator (rounding, quantization, preferred fractions).

- `{col}_fraction` - fractional part.
- `{col}_digit{k}` - **positional digit decomposition**: the base-10 digit at each position `k` in `DIGIT_POSITIONS` (`k<0` = decimals, `k=0` = units, `k>0` = tens/hundreds/...).
- `{col}_frac100` - first two decimals as an integer; `{col}_mod100` — integer part modulo 100 (two-digit views, kept since they are not single digits).
- `{col}_round` - **adaptive magnitude-based rounding**: the value snapped to a precision chosen from its column's max magnitude (`<10` → 3 decimals, `<100` → 2, else 1), mirroring the generator's likely quantization. Emitted as a new column so the raw value is preserved.
- `{col}_res_1_2 / 1_4 / 1_5 / 1_10` - residual distance of the fraction to the nearest multiple of 1/2, 1/4, 1/5, 1/10 (detects values that snap to "nice" fractions).
- `{col}_is_round` - flag for values whose fraction is ~0 or ~1.
- `{col}_mod10_d1`- string combinations, now rebuilt from the positional digits (`digit-1`/`digit-2` = first/second decimal, `digit0` = units), kept as categorical interaction features.

In [27]:
# Fractions whose nearest multiple we measure the residual distance to.
RESIDUAL_DENOMINATORS = {'1_2': 0.5, '1_4': 0.25, '1_5': 0.2, '1_10': 0.1}

# Base-10 positions to extract a digit from: k<0 -> decimals, k=0 -> units,
# k>0 -> tens/hundreds/... The string combos below assume -2, -1 and 0 are in
# the range, so keep those if you edit it.
DIGIT_POSITIONS = range(-4, 4)


def add_decimal_features(train_x, test_x, numeric_cols, positions=DIGIT_POSITIONS):
    print("Adding decimal features...")
    # Per-column rounding precision for the adaptive-round feature, decided from
    # the combined train+test magnitude (small ranges keep more decimals).
    max_abs = {c: pd.concat([train_x[c], test_x[c]]).abs().max() for c in numeric_cols}
    precision = lambda m: 3 if m < 10 else 2 if m < 100 else 1

    def mutate(df):
        for col in numeric_cols:
            x = df[col]
            floor_x = np.floor(x)
            frac = np.round(x - floor_x, 5)

            df[f"{col}_fraction"] = frac

            # Positional digit decomposition: the digit at each base-10 position
            # (generalises the old d1/d2/mod10 single-digit features). The inner
            # round kills float noise before flooring (e.g. 3.7/0.1 -> 37, not 36).
            for k in positions:
                df[f"{col}_digit{k}"] = (np.floor(np.round(x / 10.0**k, 6)) % 10).astype('int8')

            # Two-digit views (not single digits, so kept) + adaptive rounding.
            df[f"{col}_frac100"] = np.floor(frac * 100).astype(int)
            df[f"{col}_mod100"] = (floor_x % 100).astype(int)
            df[f"{col}_round"] = x.round(precision(max_abs[col]))

            for name, m in RESIDUAL_DENOMINATORS.items():
                df[f"{col}_res_{name}"] = np.round(np.abs(frac - np.round(frac / m) * m), 5)

            df[f"{col}_is_round"] = ((frac < 0.005) | (frac > 0.995)).astype(int)

            # String interaction combos, rebuilt from the positional digits:
            # digit-1/-2 = first/second decimal, digit0 = units.
            d1 = df[f"{col}_digit-1"].astype(str)
            d2 = df[f"{col}_digit-2"].astype(str)
            d0 = df[f"{col}_digit0"].astype(str)
            df[f"{col}_mod10_d1"] = d0 + '_' + d1

    return apply_to_both(train_x, test_x, mutate)

### Bigram (categorical interaction) features `add_bigram_features`

Builds every **pairwise combination of categorical columns** by concatenating their string values (`col1__col2 = "valA_valB"`). This captures co-occurrence patterns that single categories miss (e.g. a crop type that only needs irrigation in a specific season).

The result is a set of new high-cardinality **string** columns, emitted as-is for the downstream gradient-boosting model to encode natively (CatBoost / LightGBM consume categorical columns directly). Note that `frequency_encode` and `count_ratio_encode` in this notebook operate on the *raw* `categorical_cols` captured before feature engineering, so they do **not** re-encode these bigrams.

In [28]:
def add_bigram_features(train_x, test_x, categorical_cols):
    print("Adding bigram features...")
    def mutate(df):
        for i, col1 in enumerate(categorical_cols):
            for col2 in categorical_cols[i + 1:]:
                df[f"{col1}__{col2}"] = df[col1].astype(str) + "_" + df[col2].astype(str)

    return apply_to_both(train_x, test_x, mutate)

### Frequency encoding `frequency_encode`

Replaces each category with its **relative frequency** (`value_counts(normalize=True)`), giving the model a numeric "how common is this value" signal. The statistic is computed over `train + test` combined, so rare/common levels are estimated from all available rows.

> Note: this is *transductive* so it uses the test distribution. That is acceptable for a fixed Kaggle test set, but the frequency map would need to be **fitted on train only and saved** to apply this to genuinely new data later.

In [29]:
def frequency_encode(train_x, test_x, categorical_cols):
    print("Adding frequency-encoded features...")
    # Frequencies estimated over train+test combined (transductive — fine for a
    # fixed Kaggle test set; for genuinely new data, fit on train only and save).
    freq_maps = {
        col: pd.concat([train_x[col], test_x[col]]).value_counts(normalize=True)
        for col in categorical_cols
    }

    def mutate(df):
        for col, freq in freq_maps.items():
            df[f"freq_{col}"] = np.round(df[col].map(freq), 3)

    return apply_to_both(train_x, test_x, mutate)

### Radix encoding `radix_encode`

Packs **groups of related features into a single integer code** by placing each at a different "digit position" (multiplying by powers of 10), e.g. `Soil_pH * 100 + soil_type_index * 1000`. This gives the model compact, explicit interaction features (soil pH × soil type, crop × growth stage, temperature × humidity, irrigation × water source, field area × mulching).

The interactions are not hardcoded in the function they are read from the `RADIX_FEATURES` config (a list of terms per output column). Categorical terms are mapped to a deterministic `category → index` code built per column over `train + test` with `np.sort`, so the same category always gets the same code in both sets (sorting — rather than appearance order, which differs between train and test — keeps the two encodings aligned). To change which features get combined, edit the config, not this cell.

In [30]:
def radix_encode(train_x, test_x, radix_features):
    print("Adding radix features...")
    # Every categorical column referenced by the specs needs a deterministic
    # category -> index map, shared across train+test (sorted) so a category
    # always gets the same code in both sets.
    cat_cols = {
        term['col']
        for spec in radix_features.values()
        for term in spec['terms']
        if term['kind'] == 'cat'
    }
    code_maps = {
        col: {val: idx for idx, val in enumerate(
            np.sort(pd.concat([train_x[col], test_x[col]]).dropna().unique())
        )}
        for col in cat_cols
    }

    def term_value(df, term):
        if term['kind'] == 'cat':
            values = df[term['col']].map(code_maps[term['col']])
        else:
            values = df[term['col']]
        values = values * term['mult']
        return values.astype(int) if term.get('cast_int') else values

    def mutate(df):
        for name, spec in radix_features.items():
            total = sum(term_value(df, term) for term in spec['terms'])
            if spec['round'] is not None:
                total = np.round(total, spec['round'])
            df[f'radix_{name}'] = total

    return apply_to_both(train_x, test_x, mutate)

## Group 2: features that require the labeled original dataset

The functions below compare each row against the `original_dataset` the competition data was generated from, so they run only when `original_data` is available (`original_data is not None`).

### Original-data KNN lookup `original_data_lookup`

Treats the **labeled original dataset** as a reference table and asks, for every competition row, "which original rows look most like me, and what was their label?"

Steps:
1. Scale numeric columns (`StandardScaler`) and one-hot encode categoricals (`OneHotEncoder`), fitted **on the original data**.
2. Transform train/test with the *same* fitted scaler/encoder and stack into a common feature matrix.
3. Build a `cKDTree` over the original matrix and query the **3 nearest neighbors** for each row.

New features:
- `nearest_original_label_mode` - majority `TARGET` label (`Irrigation_Need` here) among the 3 nearest original rows: a KNN prediction used as a feature.
- `distance_to_nearest_min` - distance to the single closest original row.
- `distance_to_nearest_mean` - mean distance across the 3 neighbors (a "how typical is this row" proxy).

No target leakage from the rows themselves: labels come from the *original* dataset, not from `train_y`.

In [31]:
def original_data_lookup(train_x, test_x, original_data, numeric_cols, categorical_cols, target):
    print("Adding original data lookup features...")
    orig_features = original_data.drop(columns=[target])
    scaler = StandardScaler()
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # Fit the scaler/encoder on the original data and build the reference matrix.
    orig_matrix = np.hstack([
        scaler.fit_transform(orig_features[numeric_cols]),
        encoder.fit_transform(orig_features[categorical_cols]),
    ])
    tree = cKDTree(orig_matrix)
    orig_targets = original_data[target].to_numpy()

    def lookup(df):
        # Transform with the same fitted scaler/encoder, then query 3 NNs.
        matrix = np.hstack([
            scaler.transform(df[numeric_cols]),
            encoder.transform(df[categorical_cols]),
        ])
        distances, indices = tree.query(matrix, k=3)
        neighbor_labels = orig_targets[indices]

        df['nearest_original_label_mode'] = pd.DataFrame(neighbor_labels).mode(axis=1)[0].to_numpy()
        df['distance_to_nearest_min'] = np.round(distances[:, 0], 2)
        df['distance_to_nearest_mean'] = np.round(distances.mean(axis=1), 2)

    return apply_to_both(train_x, test_x, lookup)

### Snap features `add_snap_features`

For every numeric column, each value is **snapped to the nearest value that actually exists in the original dataset**, and the absolute distance to it is stored.

- `{col}_snap` - the closest real value from the original data.
- `{col}_snap_diff` - how far the synthetic value drifted from that real value.

Intuition: synthetic rows that sit almost exactly on an original value (tiny `snap_diff`) were likely lightly perturbed and may carry a strong signal, whereas large diffs indicate heavily altered values. The lookup uses a sorted array + `searchsorted`, comparing the candidate and its predecessor to pick the truly nearest neighbor.

In [32]:
def add_snap_features(train_x, test_x, original_data, features_to_snap):
    print("Adding snap features...")
    # Sorted unique original values per column, for nearest-neighbour snapping.
    snap_lookups = {
        col: np.sort(original_data[col].dropna().unique())
        for col in features_to_snap
    }

    def nearest(sorted_vals, values):
        idx = np.clip(np.searchsorted(sorted_vals, values, side='left'), 0, len(sorted_vals) - 1)
        idx_prev = np.clip(idx - 1, 0, len(sorted_vals) - 1)
        closer = np.where(
            np.abs(values - sorted_vals[idx]) <= np.abs(values - sorted_vals[idx_prev]),
            idx, idx_prev,
        )
        return sorted_vals[closer]

    def mutate(df):
        for col, sorted_vals in snap_lookups.items():
            values = df[col].to_numpy()
            snapped = nearest(sorted_vals, values)
            df[f"{col}_snap"] = snapped
            df[f"{col}_snap_diff"] = np.round(np.abs(values - snapped), 5)

    return apply_to_both(train_x, test_x, mutate)

### Count-ratio encoding `count_ratio_encode`

For each category, computes the ratio of its **count in the competition data** (`train + test`) to its **count in the original dataset**:

```
count_ratio = count_in(train+test) / count_in(original)
```

This measures how strongly each category was **over- or under-sampled** when the synthetic dataset was generated from the original — a direct signal about the generation process. Categories present in the competition data but absent from the original produce `NaN` (worth a `fillna` if it matters downstream).

In [33]:
def count_ratio_encode(train_x, test_x, categorical_cols, original_data):
    print("Adding count ratio features...")
    # Ratio of each category's count in train+test vs. in the original dataset
    # (how strongly it was over-/under-sampled by the synthetic generator).
    ratio_maps = {
        col: (
            pd.concat([train_x[col], test_x[col]]).value_counts()
            / original_data[col].value_counts()
        )
        for col in categorical_cols
    }

    def mutate(df):
        for col, ratio in ratio_maps.items():
            df[f"count_ratio_{col}"] = np.round(df[col].map(ratio), 3)

    return apply_to_both(train_x, test_x, mutate)

## Group 3: target-aware features (require `train_y`)

Unlike Groups 1 and 2, these features use the **target**, so they deliberately break the target-free contract of the rest of the pipeline: they are *fit on train and applied to test*, and within `train` each row must avoid leaking its own label.

### Ordered Target Encoding `ordered_target_encode`

For every categorical column, replaces the category with the **per-class probability of the target given that category**, smoothed toward the global class prior — i.e. a small KNN-like "what does this category usually predict" signal, one column per class (`{col}_TE_cls{class}`).

Leakage is controlled two ways:

- **Train** uses an *ordered/cumulative* scheme (`OrderedTE`): each row is encoded from only the rows that came before it, so its own label never leaks in. Because the result depends on row order, and because preprocessing must keep exactly one row per `id` (aligned with `train_y`) rather than stacking shuffled copies like the modeling notebook, we **average the ordered encoding over several shuffles** (`TE_N_SHUFFLES`) and reindex back to the original rows — this cancels most of the ordering variance while preserving alignment.
- **Test** is encoded from the **full-train** per-category statistics; categories unseen in train fall back to the global class prior.

`TE_SMOOTHING` controls how strongly rare categories are pulled toward the prior. Set the encoded columns via `TE_COLS` in the config (defaults to the detected `categorical_cols`); add the high-cardinality bigram columns there if you want them target-encoded too.

In [34]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=(), target_col='target'):
        self.train = train
        self.target_col = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train[target_col].unique())
        # Prior P(class), aligned by class with self.classes_.
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []
            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(int)

                df = train[[c]].copy()
                df['y'] = y_binary.values
                df['cnt'] = 1
                # Cumulative count/sum per category, EXCLUDING the current row.
                df['cum_cnt'] = df.groupby(c)['cnt'].cumsum() - df['cnt']
                df['cum_sum'] = df.groupby(c)['y'].cumsum() - df['y']

                smooth_prior = self.a * self.global_prior_[k]
                te_col = f'{c}_TE_cls{cls}'
                df[te_col] = (df['cum_sum'] + smooth_prior) / (df['cum_cnt'] + self.a)
                # First time a category appears -> no history, use the prior.
                df.loc[df['cum_cnt'] == 0, te_col] = self.global_prior_[k]
                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)['y'].agg(['count', 'sum']).reset_index()
                stats_df.columns = [c, f'{c}_count_cls{cls}', f'{c}_sum_cls{cls}']
                stats_df[f'{c}_prior_cls{cls}'] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how='outer')
            setattr(self, f'{c}_stats', combined_stats)

        return self.train

    def transform(self, test):
        for c in self.category_cols:
            stats_df = getattr(self, f'{c}_stats')
            test = test.merge(stats_df, on=c, how='left')
            for k, cls in enumerate(self.classes_):
                te_col = f'{c}_TE_cls{cls}'
                count_col = f'{c}_count_cls{cls}'
                sum_col = f'{c}_sum_cls{cls}'
                prior_col = f'{c}_prior_cls{cls}'
                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * self.global_prior_[k]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(self.global_prior_[k])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test[te_col] = self.global_prior_[k]
        return test

In [35]:
def ordered_target_encode(train_x, test_x, train_y, category_cols, target_col,
                          n_shuffles=4, smoothing=1, seed=2026):
    print("Adding ordered target encoding features...")
    full = train_x[category_cols].copy()
    full[target_col] = train_y.to_numpy()

    # Full-train fit: per-category stats + class priors (for test + NaN fill).
    te = OrderedTE(a=smoothing)
    te.fit(full.copy(), category_cols=category_cols, target_col=target_col)
    n_cls = len(te.classes_)

    # Train: average the ordered TE over shuffles, realigned to original rows.
    te_sum = None
    for i in range(n_shuffles):
        shuffled = full.sample(frac=1, random_state=seed + i)
        enc = OrderedTE(a=smoothing).fit(shuffled, category_cols=category_cols, target_col=target_col)
        cols = [c for c in enc.columns if '_TE_cls' in c]
        enc = enc[cols].reindex(full.index)
        te_sum = enc if te_sum is None else te_sum + enc
    train_te = te_sum / n_shuffles

    # Test: encode from the full-train statistics (order-independent).
    test_enc = te.transform(test_x[category_cols].copy())
    cols = [c for c in test_enc.columns if '_TE_cls' in c]
    test_te = test_enc[cols]
    test_te.index = test_x.index

    # Fill unseen-category gaps with the matching per-class global prior.
    prior_fill = pd.Series({col: te.global_prior_[j % n_cls] for j, col in enumerate(cols)})
    train_te = train_te.fillna(prior_fill).round(5)
    test_te = test_te.fillna(prior_fill).round(5)
    test_te = test_te.reindex(columns=train_te.columns)  # identical column order

    return (pd.concat([train_x, train_te], axis=1),
            pd.concat([test_x, test_te], axis=1))

## Cleanup drop constant columns

After all groups have run, some engineered columns can be constant across the whole dataset - e.g. a high digit position (`digit3`) that is always `0` for a small-magnitude column, or a residual that never fires. They carry no signal and only bloat the matrix; with column subsampling (`colsample_bytree < 1`) they also dilute the useful features. `drop_constant_columns` removes any column with a single unique value across `train + test` and returns the list it dropped.

In [36]:
def drop_constant_columns(train_x, test_x):
    print("Dropping constant columns...")
    # Columns with a single unique value across train+test carry no signal and
    # only bloat the matrix (and dilute column subsampling). Returns the trimmed
    # frames plus the list of dropped names.
    constant = [c for c in test_x.columns
                if pd.concat([train_x[c], test_x[c]]).nunique() <= 1]
    return train_x.drop(columns=constant), test_x.drop(columns=constant), constant

### Applying the pipeline

Group 1 runs first (it never needs `original_data`), then Group 2 runs only if the original dataset was loaded (`original_data is not None`). Every function reads the **raw** `numeric_cols` / `categorical_cols` lists captured before any feature engineering, so each step ignores columns added by earlier steps and both `train_x` and `test_x` end with an identical schema. Ordering between the groups is therefore safe - e.g. the KNN matrix in `original_data_lookup` is built only from the raw original feature space regardless of when it runs.

Group 3 (Ordered Target Encoding) runs last. It is the **only** step that uses `train_y`: it is fit on train and applied to test, so unlike Groups 1 and 2 it is not symmetric across the two sets (train gets the leakage-safe ordered encoding, test gets the full-train statistics). It still reads the raw `categorical_cols` (via `te_cols`), so it is order-independent w.r.t. the earlier groups too.

In [37]:
# --- Group 1: no original dataset required ---
train_x, test_x = add_decimal_features(train_x, test_x, numeric_cols)
train_x, test_x = add_bigram_features(train_x, test_x, categorical_cols)
train_x, test_x = frequency_encode(train_x, test_x, categorical_cols)
train_x, test_x = radix_encode(train_x, test_x, RADIX_FEATURES)

# --- Group 2: requires the labeled original dataset ---
if original_data is not None:
    train_x, test_x = original_data_lookup(train_x, test_x, original_data, numeric_cols, categorical_cols, TARGET)
    train_x, test_x = add_snap_features(train_x, test_x, original_data, numeric_cols)
    train_x, test_x = count_ratio_encode(train_x, test_x, categorical_cols, original_data)

# --- Group 3: target-aware (Ordered Target Encoding) ---
te_cols = TE_COLS or categorical_cols
train_x, test_x = ordered_target_encode(
    train_x, test_x, train_y, te_cols, TARGET,
    n_shuffles=TE_N_SHUFFLES, smoothing=TE_SMOOTHING, seed=TE_SEED,
)

# --- Cleanup: drop columns that ended up constant across train+test ---
train_x, test_x, dropped = drop_constant_columns(train_x, test_x)
print(f"Dropped {len(dropped)} constant columns: {dropped}")

C:\Users\Evgenii\AppData\Local\Temp\ipykernel_16708\1130053544.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_res_{name}"] = np.round(np.abs(frac - np.round(frac / m) * m), 5)
C:\Users\Evgenii\AppData\Local\Temp\ipykernel_16708\1130053544.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_is_round"] = ((frac < 0.005) | (frac > 0.995)).astype(int)
C:\Users\Evgenii\AppData\Local\Temp\ipykernel_16708\1130053544.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of ca

Adding decimal features...
Adding bigram features...
Adding frequency-encoded features...
Adding radix features...
Adding original data lookup features...
Adding snap features...
Adding count ratio features...
Adding ordered target encoding features...


C:\Users\Evgenii\AppData\Local\Temp\ipykernel_16708\1130053544.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_mod10_d1"] = d0 + '_' + d1
C:\Users\Evgenii\AppData\Local\Temp\ipykernel_16708\1130053544.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_fraction"] = frac
C:\Users\Evgenii\AppData\Local\Temp\ipykernel_16708\1130053544.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

Dropping constant columns...
Dropped 52 constant columns: ['Soil_pH_digit-4', 'Soil_pH_digit-3', 'Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_pH_is_round', 'Soil_Moisture_digit-4', 'Soil_Moisture_digit-3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Soil_Moisture_is_round', 'Organic_Carbon_digit-4', 'Organic_Carbon_digit-3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Organic_Carbon_is_round', 'Electrical_Conductivity_digit-4', 'Electrical_Conductivity_digit-3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit-4', 'Temperature_C_digit-3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Temperature_C_is_round', 'Humidity_digit-4', 'Humidity_digit-3', 'Humidity_digit2', 'Humidity_digit3', 'Humidity_is_round', 'Rainfall_mm_digit-4', 'Rainfall_mm_digit-3', 'Rainfall_mm_is_round', 'Sunlight_Hours_digit-4', 'Sunlight_Hours_digit-3', 'Sunlight_Hours_digit2', 'Sunlight

## Output

`train_x.tail()` is a quick check. The final cell writes the preprocessed train/test features to `./data/`.

In [38]:
train_x.tail()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Irrigation_Type_TE_clsMedium,Water_Source_TE_clsHigh,Water_Source_TE_clsLow,Water_Source_TE_clsMedium,Mulching_Used_TE_clsHigh,Mulching_Used_TE_clsLow,Mulching_Used_TE_clsMedium,Region_TE_clsHigh,Region_TE_clsLow,Region_TE_clsMedium
id,,,,,,,,,,,,,,,,,,,,,
5,Sandy,5.09,24.70,1.28,0.48,12.21,92.35,696.03,9.11,7.37,...,0.23125,0.02917,0.82083,0.15000,0.30417,0.53333,0.16250,0.0375,0.76667,0.19583
6,Silt,7.53,49.67,1.44,1.62,14.02,61.65,889.39,8.44,16.76,...,0.29583,0.06250,0.75000,0.18750,0.17750,0.59000,0.23250,0.0375,0.85000,0.11250
7,Loamy,7.56,48.61,0.38,1.31,22.78,61.53,708.82,9.96,7.83,...,0.35000,0.31250,0.50000,0.18750,0.05417,0.78333,0.16250,0.0875,0.65000,0.26250
8,Silt,6.02,53.01,0.90,0.49,20.55,61.30,1536.36,10.50,19.38,...,0.13750,0.05208,0.60417,0.34375,0.04792,0.72500,0.22708,0.1000,0.60000,0.30000
9,Silt,7.39,41.91,0.58,0.78,39.25,85.52,281.76,8.86,13.13,...,0.25000,0.07083,0.71667,0.21250,0.08000,0.58000,0.34000,0.1000,0.60000,0.30000


In [39]:
# Re-attach the target so the saved train file carries the label for the
# modeling notebooks (test_x has no target, as expected).
train_x[TARGET] = train_y

In [40]:
train_x.to_csv(f'{DATA_DIR}/train_x_preprocessed_example.csv', index=True)
test_x.to_csv(f'{DATA_DIR}/test_x_preprocessed_example.csv', index=True)